In [ ]:
from impact import Impact
from impact.model.impact_model import LUMEImpactModel
from impact.model.impact_config import (
    VariableMappingConfig,
    ElementsConfig,
    StatsConfig,
    ParticlesConfig,
    RunInfoConfig,
)

from lume.variables import ScalarVariable, NDVariable, ParticleGroupVariable
import matplotlib.pyplot as plt

In [ ]:
# Load LCLS Impact-T lattice
imp = Impact("templates/lcls_injector/ImpactT.in")
imp.header["Np"] = 10000
imp.header["Nx"] = 16
imp.header["Ny"] = 16
imp.header["Nz"] = 16
imp.header["Dt"] = 5e-13

# Turn Space Charge off. Both these syntaxes work
imp.header["Bcurr"] = 0
imp["header:Bcurr"] = 0

# Other switches
imp.timeout = None

# Switches for MPI
imp.numprocs = 0  # Auto-select

imp.run()

In [ ]:
# Automatically convert to lume-model
ele_name_mappings = {"QUAD:IN20:425": "QE01"}
model = LUMEImpactModel.from_impact(
    imp,
    variable_mapping=VariableMappingConfig(
        elements=ElementsConfig(
            pattern="{name}:{attrib}",
            name_mappings=ele_name_mappings,
            regex=r"^(?P<name>.+):(?P<attrib>[^:]+)$",  # Regex to capture item after last colon as attribute, everything else as name
        ),
        stats=StatsConfig(pattern="STATS:{name}"),
        particles=ParticlesConfig(pattern="PAR:{name}"),
        run_info=RunInfoConfig(pattern="RUNINFO:{key}"),
    ),
    dummy_run=True,
)

# Show default loaded variables, can control names and inclusion of all attributes from `VariableMappingConfig` object
[var for var in model.vars if "IN20" in var.name]

In [ ]:
# Look at the stats keys
[
    (var.name, var.shape)
    for var in model.vars
    if isinstance(var, NDVariable) and var.read_only
]

In [ ]:
# Look at the particles keys
[var.name for var in model.vars if isinstance(var, ParticleGroupVariable)]

In [ ]:
# Add custom variable for each quad (SLAC BCTRL)
inv_name_map = {v: k for k, v in ele_name_mappings.items()}
ctrl_vars = []
for ele in imp.lattice:
    name = ele["name"]
    if ele["type"] == "quadrupole":
        ctrl_vars.append(
            ScalarVariable(name=inv_name_map.get(name, name) + ":BCTRL", unit="kG")
        )
model.vars.extend(ctrl_vars)


# Add custom handlers using element matching
@model.transformer.ele_setter(control_attrib="BCTRL", ele_type="quadrupole")
def set_bctrl(
    imp: Impact,
    value: float,
    control_name: str,
    tool_name: str,
    control_attrib: str,
    tool_attrib: str,
):
    b1_grad = value / imp.ele[tool_name]["L"]  # Example calc, use own model to convert
    print(f"Setting {control_name} ({tool_name}) b1_grad = {b1_grad:.2f}")
    imp.ele[tool_name]["b1_gradient"] = b1_grad


@model.transformer.ele_getter(control_attrib="BCTRL", ele_type="quadrupole")
def get_bctrl(
    imp: Impact,
    control_name: str,
    tool_name: str,
    control_attrib: str,
    tool_attrib: str,
):
    integral = (
        imp.ele[tool_name]["b1_gradient"] * imp.ele[tool_name]["L"]
    )  # Example calc, use own model to convert
    print(f"Read {control_name} ({tool_name}) integral = {integral:.2f}")
    return integral


# Try using the setter
model.set({"QUAD:IN20:425:BCTRL": 0.0})

In [ ]:
# Get the stats object
stats = model.get(["STATS:sigma_x", "STATS:mean_z", "STATS:mean_kinetic_energy"])
plt.plot(stats["STATS:mean_z"], 1e6 * stats["STATS:sigma_x"], c="C0")
plt.xlabel("s (m)")
plt.ylabel("RMS Beam Size (um, Blue)")
plt.twinx()
plt.plot(stats["STATS:mean_z"], 1e-6 * stats["STATS:mean_kinetic_energy"], c="C1")
plt.ylabel("Beam Energy (MeV, Orange)")